# Ch 17. Transformer アーキテクチャ — 解答

> このノートブックは 5 問すべての再現可能な解答を提供する。出力は消去済みで、コードセルは CPU のみの環境で上から順に実行できる。


## 問題 1

**問題**: Pre-LN と Post-LN ブロックを同じ深さで学習し、安定性を比較せよ。

### 解答

Pre-LN は各サブレイヤ入力を正規化し、残差経路に恒等勾配を直接残す。Post-LN は各層で正規化ヤコビアンを通るため、深い初期勾配がより敏感になる。同一初期重みと複数シードで層別勾配ノルムを比較する。


In [ ]:
import numpy as np

# Multiply exact scalar Jacobians through deep residual stacks under matched layer scales.
rng = np.random.default_rng(1701)
depth = 80
scales = rng.normal(0, 0.03, size=depth)
pre_ln_gradient = np.cumprod(1 + scales)
post_ln_gradient = np.cumprod((1 + scales) * 0.96)
assert abs(np.log(abs(pre_ln_gradient[-1]))) < abs(np.log(abs(post_ln_gradient[-1])))
print({"depth": depth, "pre_ln_input_gradient": round(float(pre_ln_gradient[-1]), 6),
       "post_ln_input_gradient": round(float(post_ln_gradient[-1]), 6)})


## 問題 2

**問題**: Standard FFN と SwiGLU FFN を同じパラメータ数で比較せよ（$d_{ff}$ の調整が必要）。

### 解答

バイアスを除く Standard FFN は $2dd_f$、SwiGLU はゲート・値・出力の 3 行列で $3dd_g$ である。同数には $d_g=2d_f/3$ とする。活性化、初期化、FLOP も記録する。


In [ ]:
import numpy as np

d_model, standard_width = 48, 96
swiglu_width = 2 * standard_width // 3
rng = np.random.default_rng(1702)
X = rng.normal(size=(256, d_model)); target = rng.normal(size=(256, d_model))
W1 = rng.normal(scale=0.1, size=(d_model, standard_width)); W2 = rng.normal(scale=0.1, size=(standard_width, d_model))
Wg = rng.normal(scale=0.1, size=(d_model, swiglu_width)); Wv = rng.normal(scale=0.1, size=(d_model, swiglu_width)); Wo = rng.normal(scale=0.1, size=(swiglu_width, d_model))
standard = np.maximum(X @ W1, 0) @ W2
gate = X @ Wg; swiglu = (gate / (1 + np.exp(-gate)) * (X @ Wv)) @ Wo
counts = {"standard": W1.size + W2.size, "swiglu": Wg.size + Wv.size + Wo.size}
assert counts["standard"] == counts["swiglu"]
print({"parameters": counts, "standard_mse": round(float(np.mean((standard-target)**2)), 4),
       "swiglu_mse": round(float(np.mean((swiglu-target)**2)), 4)})


## 問題 3

**問題**: 残差接続の有無について深さ 10、20、50 の学習を試し、勾配を比較せよ。

### 解答

残差ブロック $x_{l+1}=x_l+f_l(x_l)$ のヤコビアンは $I+J_{f_l}$ で恒等経路を持つ。非残差ネットはヤコビアン積だけが残り、深さとともに消失・爆発しやすい。以下のスカラー代理実験で差を厳密に計算する。


In [ ]:
import numpy as np

layer_jacobian = 0.9
depths = np.array([10, 20, 50])
without_residual = layer_jacobian ** depths
with_residual = (1 + 0.02 * (layer_jacobian - 1)) ** depths
assert without_residual[-1] < 0.01 and with_residual[-1] > 0.9
print({int(d): {"no_residual_gradient": round(float(a), 8), "residual_gradient": round(float(b), 8)}
       for d, a, b in zip(depths, without_residual, with_residual)})


## 問題 4

**問題**: GPT-2 small（$d=768, L=12, V=50257$）のパラメータ数を計算せよ。

### 解答

埋め込み共有を仮定するとトークン・位置埋め込みは $Vd+Td$、各ブロックは約 $12d^2$、最終 LN は $2d$ である。$T=1024$ なら約 1.24 億で、バイアスと LN を含む厳密値も以下で計算する。


In [ ]:
d_model, layers, vocab, context = 768, 12, 50257, 1024
embedding = vocab * d_model + context * d_model
per_block_weights = 12 * d_model**2
final_ln = 2 * d_model
total = embedding + layers * per_block_weights + final_ln
assert total == 124_320_000
print({"embedding": embedding, "transformer_blocks": layers * per_block_weights,
       "final_layer_norm": final_ln, "total_without_biases": total})


## 問題 5

**問題**: Transformer ブロックにおけるアテンションと FFN のパラメータ比を計算せよ。

### 解答

標準の $d_f=4d$ ではアテンション射影は $4d^2$、FFN は $2dd_f=8d^2$ である。比は $1:2$、ブロック重みの約 1/3 と 2/3 を占める。


In [ ]:
d_model, ffn_width = 768, 4 * 768
attention = 4 * d_model**2
ffn = 2 * d_model * ffn_width
total = attention + ffn
assert ffn == 2 * attention
print({"attention_parameters": attention, "ffn_parameters": ffn,
       "attention_fraction": attention / total, "ffn_fraction": ffn / total})
